user_task_submissions.csv — основная история решений по задачам
user_practice_submissions.csv — история КИМов / практик
task_competence_mapping.csv — задача -> компетенция
micromodules_competencies.csv — микромодуль -> компетенция

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
tasks = pd.read_csv("submissions_archive/user_task_submissions.csv", sep=";")
practices = pd.read_csv("submissions_archive/user_practice_submissions.csv", sep=";")
mapping = pd.read_csv("submissions_archive/task_competence_mapping.csv", sep=";")
micro = pd.read_csv("submissions_archive/micromodules_competencies.csv", sep=";")

tasks["submission_time"] = pd.to_datetime(tasks["submission_time"], errors="coerce")
tasks["is_correct"] = tasks["is_correct"].astype(str).str.lower().map({"true": 1, "false": 0})
tasks["score"] = pd.to_numeric(tasks["score"], errors="coerce")
tasks["max_score"] = pd.to_numeric(tasks["max_score"], errors="coerce")

practices["submission_time"] = pd.to_datetime(practices["submission_time"], errors="coerce")

In [ ]:
task_vectors = (
    mapping[["task_id", "competence_code"]]
    .drop_duplicates()
    .assign(v=1)
    .pivot(index="task_id", columns="competence_code", values="v")
    .fillna(0)
    .astype(int)
)
task_vectors

competence_code,1.0.0. Планиметрия,10.0.0,11.0.0,12.12.1,12.13.5,12.17.2,12.27.1,12.27.8,12.34.1,12.34.2,...,9.1.6.,9.14.6.,9.15.4.,9.16.1.,9.16.5.,9.2.1.,9.3.2.,9.4.4.,9.5.6.,9.6.6.
task_id,,,,,,,,,,,,,,,,,,,,,
00bedc06-de2b-4183-9851-ef89cfaef03f,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
01bd4928-c112-476f-8292-9d09eb68441e,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
01deabf3-e24a-47b4-8e7b-02c6fdbae0a6,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
0272d631-1845-4109-a797-adcd8db0f500,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
02b1172e-dd7a-4384-b49e-98ea85d80ba0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fe6a781f-5fcd-420d-b440-f40c9474656d,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
fe98ebef-f15e-4425-bdec-7555085668f7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
fe9b66eb-09e6-4075-b178-2cdbb6a3d396,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
user_number = 13

trajectory = (
    tasks[tasks["user_number"] == user_number]
    .sort_values("submission_time")
    .reset_index(drop=True)
)
trajectory

,user_number,submission_id,practice_submission_id,task_id,task_name,task_type,max_score,node_id,node_name,course_id,task_number,status,score,autochecked,submission_time,is_correct
0,13,8c07e467-f8bb-4f7d-a440-d477093bb9a6,dbcd6a3c-59a1-4370-91bb-ab8fcdc42b5c,0c8f1c8e-ddf1-4061-8478-f0ae44ef3b3b,№30 - ФИПИ - AD5F64,SHORT_TEXT,1.0,a6cd3ec7-c34a-4688-898c-a039d2c16c0a,Номер 1. Планиметрия,6f2007d9-b79f-42aa-93d6-9e612161d464,1,ACCEPTED,1,True,2026-02-18 18:50:37.741040+00:00,1
1,13,c1678a4e-b435-4d9d-a565-a6329098b957,dbcd6a3c-59a1-4370-91bb-ab8fcdc42b5c,8262e2f7-531f-4098-b384-c1da5adb2173,№2 - ФИПИ - 8e0D68,SHORT_TEXT,1.0,8ec1ca72-624f-4020-a24d-9078da1b128b,Импортированные задачи ФИПИ,bc1d4037-e5be-4ea4-94d8-e812a1343379,2,ACCEPTED,1,True,2026-02-18 18:50:37.741040+00:00,1
2,13,fcf3c6ef-0d68-4fc3-a03f-d50912755a8a,dbcd6a3c-59a1-4370-91bb-ab8fcdc42b5c,857b5600-79f5-4463-a9dc-5848bfd6d02c,№15 - ФИПИ - E4F70F,SHORT_TEXT,1.0,20d15d70-4d6b-47d1-b782-255b6035b985,22. Решение логарифмических уравнений с одинак...,830dd404-3b3f-4a21-af65-5ddbf150b6ed,6,ACCEPTED,1,True,2026-02-18 18:50:37.741040+00:00,1
3,13,cd47ad3a-2524-4204-9b51-2a08d47437b8,dbcd6a3c-59a1-4370-91bb-ab8fcdc42b5c,367528c5-687b-4198-af7e-4f335ccb86bf,№12 - ФИПИ - 941D07,SHORT_TEXT,1.0,d0700977-3f66-49e5-b561-fbb92f53c794,1. Классическое определение вероятности: равно...,0be1191d-2a75-4408-90c6-e45ea709e43e,4,ACCEPTED,1,True,2026-02-18 18:50:37.741040+00:00,1
4,13,2c899cd4-5ebb-44f6-ab4f-5345c9243ed7,dbcd6a3c-59a1-4370-91bb-ab8fcdc42b5c,c3f7b08c-6390-40ca-906e-ddac47f717f4,№23 - ФИПИ - 467AAF,SHORT_TEXT,1.0,cb8d3596-0857-4bc5-addd-96612fd11596,Номер 3 ЕГЭ,d00a64e6-a938-4c69-8b91-bc754bdcb80e,3,ACCEPTED,1,True,2026-02-18 18:50:37.741040+00:00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,13,92504314-ee7e-4229-a7e1-25ac668747fa,7cabdb71-a0dc-4745-ac3f-fd73181152cd,def2a7e6-b489-40d1-85ee-2fc42e91a915,№1 - ФИПИ - 8D3587,SHORT_TEXT,1.0,37a505a9-3070-4dc1-95b4-d0a367a5f5fd,Импортированные задачи ФИПИ,6f2007d9-b79f-42aa-93d6-9e612161d464,1,ACCEPTED,1,True,2026-03-19 12:37:47.564020+00:00,1
481,13,59283be8-c138-471f-96bf-4e798417ae3f,7cabdb71-a0dc-4745-ac3f-fd73181152cd,43193c53-01a0-4c09-a7de-707e53cd2a68,№28 - ФИПИ - 01BE6F,SHORT_TEXT,1.0,a6cd3ec7-c34a-4688-898c-a039d2c16c0a,Номер 1. Планиметрия,6f2007d9-b79f-42aa-93d6-9e612161d464,1,REJECTED,0,True,2026-03-19 12:37:47.564020+00:00,0
482,13,df51b0fe-cf8b-436a-89c3-8fe5cf030516,7cabdb71-a0dc-4745-ac3f-fd73181152cd,52702796-c6bc-46b6-89a6-0441f1afa5a9,№23 - ФИПИ - FFE399,SHORT_TEXT,1.0,a6cd3ec7-c34a-4688-898c-a039d2c16c0a,Номер 1. Планиметрия,6f2007d9-b79f-42aa-93d6-9e612161d464,1,REJECTED,0,True,2026-03-19 12:37:47.564020+00:00,0
483,13,6f0366b8-085f-42ac-99b2-c7560f29f14e,7cabdb71-a0dc-4745-ac3f-fd73181152cd,f9a4ce8f-c8a6-49c1-a793-935dd6ff319c,№8 - ФИПИ - 84E9F2,SHORT_TEXT,1.0,a6cd3ec7-c34a-4688-898c-a039d2c16c0a,Номер 1. Планиметрия,6f2007d9-b79f-42aa-93d6-9e612161d464,1,ACCEPTED,1,True,2026-03-19 12:37:47.564020+00:00,1


In [ ]:
def compute_trajectory_metrics(user_df, task_vectors):
    seq = user_df[
        ["task_id", "submission_time", "is_correct", "score", "max_score", "task_number", "node_name"]
    ].copy()

    seq = seq.drop_duplicates(subset=["submission_time", "task_id"]).reset_index(drop=True)

    vecs = task_vectors.reindex(seq["task_id"]).fillna(0).values

    if len(seq) < 2:
        return None

    step_distances = [
        cosine_distances(vecs[i:i+1], vecs[i+1:i+2])[0, 0]
        for i in range(len(vecs) - 1)
    ]

    dominant_comp = (
        task_vectors.reindex(seq["task_id"])
        .fillna(0)
        .idxmax(axis=1)
        .fillna("unknown")
    )

    competence_switch_rate = (dominant_comp.shift(1) != dominant_comp).iloc[1:].mean()

    seen = set()
    returns = 0
    for c in dominant_comp:
        if c in seen:
            returns += 1
        seen.add(c)
    return_rate = returns / len(dominant_comp)

    # 5. Энтропия покрытия
    counts = dominant_comp.value_counts(normalize=True)
    entropy = -(counts * np.log2(counts + 1e-12)).sum()

    return {
        "n_steps": len(step_distances),
        "mean_step_distance": float(np.mean(step_distances)), #Средняя длина шага между соседними задачами
        "std_step_distance": float(np.std(step_distances)),
        "unique_tasks": int(seq["task_id"].nunique()),
        "return_rate": float(return_rate),#Как часто ученик возвращается в уже посещённые компетенции
        "competence_switch_rate": float(competence_switch_rate),#Как часто ученик меняет основную компетенцию
        "coverage_entropy": float(entropy),#Насколько широко и равномерно пользователь покрывает разные компетенции
        "accuracy": float(seq["is_correct"].mean())#доля правильных решений
    }

metrics = compute_trajectory_metrics(trajectory, task_vectors)
print(metrics)

{'n_steps': 484, 'mean_step_distance': 0.7167417941735913, 'std_step_distance': 0.4159084817021504, 'unique_tasks': 320, 'return_rate': 0.9484536082474226, 'competence_switch_rate': 0.6962809917355371, 'coverage_entropy': 3.6173937692485376, 'accuracy': 0.7628865979381443}


Этот пользователь:

* имеет достаточно длинную и насыщенную траекторию,
* демонстрирует не линейное, а итеративное поведение,
* часто возвращается к уже посещённым компетенциям,
* работает не в одной теме, а в нескольких областях,
* при этом довольно часто переключается между ними,
* но делает это, скорее всего, не полностью случайно.

In [15]:
def random_baseline(user_df, task_vectors, n_iter=200, seed=42):
    rng = np.random.default_rng(seed)

    seq = user_df[
        ["task_id", "submission_time", "is_correct", "score", "max_score", "task_number", "node_name"]
    ].copy()

    seq = seq.drop_duplicates(subset=["submission_time", "task_id"]).reset_index(drop=True)
    task_ids = seq["task_id"].tolist()

    if len(task_ids) < 2:
        return pd.DataFrame()

    rows = []

    for _ in range(n_iter):
        shuffled = task_ids.copy()
        rng.shuffle(shuffled)

        tmp = seq.copy()
        tmp["task_id"] = shuffled

        rows.append(compute_trajectory_metrics(tmp, task_vectors))

    return pd.DataFrame(rows)

baseline = random_baseline(trajectory, task_vectors, n_iter=200)
baseline.describe()

,n_steps,mean_step_distance,std_step_distance,unique_tasks,return_rate,competence_switch_rate,coverage_entropy,accuracy
count,200.000000,200.000000,200.000000,200.0,200.000000,200.000000,200.000000,200.000000
mean,476.710000,0.930361,0.233378,320.0,0.947666,0.896229,3.622480,0.762858
std,2.249712,0.010485,0.018129,0.0,0.000246,0.013080,0.005500,0.002506
min,471.000000,0.898600,0.177393,320.0,0.947034,0.861635,3.593605,0.757895
25%,475.000000,0.924736,0.220701,320.0,0.947479,0.887493,3.619979,0.760915
50%,477.000000,0.930485,0.233922,320.0,0.947699,0.897275,3.623021,0.762940
75%,478.000000,0.936820,0.243391,320.0,0.947808,0.905462,3.626012,0.764122
max,484.000000,0.958938,0.280679,320.0,0.948454,0.926625,3.636424,0.770042


## Оценка случайности пользовательской траектории

Для проверки гипотезы о неслучайности образовательной траектории был построен **random baseline**: для исходной последовательности задач пользователя порядок прохождения был случайным образом перемешан 200 раз с сохранением исходного множества задач. Для каждой случайной перестановки рассчитывались те же метрики траектории, что и для реального поведения.

Сравнение показало, что реальная траектория пользователя существенно отличается от случайной. Средняя длина шага между соседними задачами в пространстве компетенций составила **0.717** против **0.930** у случайного baseline. Это означает, что пользователь переходит между задачами более локально и содержательно последовательно, чем это происходило бы при случайном порядке прохождения.

Аналогично, частота смены доминирующей компетенции в реальной траектории составила **0.696**, тогда как у случайного baseline она достигала **0.896**. Следовательно, пользователь значительно реже совершает бессистемные переходы между различными содержательными зонами, чем это ожидалось бы при случайном поведении.

При этом показатели покрытия пространства компетенций (`coverage_entropy`) и возвратов в ранее посещённые области (`return_rate`) оказались близкими к случайному baseline.
